# Baseline classiche per il denoising di immagini con rumore gaussiano

In questo notebook vengono valutati alcuni metodi classici di denoising su immagini degradate da rumore gaussiano additivo con deviazione standard `σ = 25`.

L'obiettivo è costruire un confronto sperimentale con i risultati ottenuti tramite Noise2Void, utilizzando gli stessi dataset e le stesse metriche di valutazione.

I metodi analizzati sono:

- filtro media;
- filtro mediano;
- BM3D.

Il confronto viene effettuato calcolando, per ogni immagine:

- PSNR tra immagine pulita e immagine rumorosa;
- SSIM tra immagine pulita e immagine rumorosa;
- PSNR tra immagine pulita e immagine denoised;
- SSIM tra immagine pulita e immagine denoised;
- guadagno in PSNR e SSIM rispetto all'immagine rumorosa.

Le immagini pulite vengono utilizzate esclusivamente per la valutazione quantitativa finale e non intervengono nel processo di denoising.

## Installazione della libreria BM3D

BM3D è un algoritmo classico di denoising particolarmente efficace per il rumore gaussiano.  
Nel notebook viene utilizzata una libreria Python già disponibile tramite `pip`.

In [1]:
!pip install -q bm3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.0/862.0 kB 4.2 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import cv2
import bm3d

from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

## Definizione dei percorsi dei dataset

In [3]:
# -----------------------------
# CLEAN DATASET
# -----------------------------

kodak_clean_dir = Path(
    "/kaggle/input/datasets/sherylmehta/kodak-dataset"
)

bsd500_clean_dir = Path(
    "/kaggle/input/datasets/balraj98/berkeley-segmentation-dataset-500-bsds500/images"
)

div2k_clean_base_dir = Path(
    "/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images"
)
# -----------------------------
# NOISY GAUSSIAN DATASET
# -----------------------------

kodak_noisy_dir = Path(
    "/kaggle/input/notebooks/antoiann/gaussiannoise/kodak_gaussian_sigma25"
)

bsd500_noisy_dir = Path(
    "/kaggle/input/notebooks/antoiann/gaussiannoise/bsd500_gaussian_sigma25"
)

div2k_noisy_base_dir = Path(
    "/kaggle/input/notebooks/antoiann/gaussiannoise/div2k_gaussian_sigma25"
)

# -----------------------------
# OUTPUT BASELINES
# -----------------------------

output_base_dir = Path("/kaggle/working/gaussian_sigma25_classical_baselines")
output_base_dir.mkdir(parents=True, exist_ok=True)

## Definizione degli split dei dataset

BSD500 è già suddiviso in `train`, `val` e `test`.  
Per la valutazione finale delle baseline viene utilizzato principalmente lo split `test`.

DIV2K, invece, è organizzato nelle cartelle `DIV2K_train_HR` e `DIV2K_valid_HR`.  
Per la valutazione finale viene usato `DIV2K_valid_HR`.

Kodak non presenta una suddivisione interna e viene usato interamente come dataset di test esterno.

In [4]:
# -----------------------------
# BSD500 split
# -----------------------------

bsd500_train_clean_dir = bsd500_clean_dir / "train"
bsd500_val_clean_dir   = bsd500_clean_dir / "val"
bsd500_test_clean_dir  = bsd500_clean_dir / "test"

bsd500_train_noisy_dir = bsd500_noisy_dir / "train"
bsd500_val_noisy_dir   = bsd500_noisy_dir / "val"
bsd500_test_noisy_dir  = bsd500_noisy_dir / "test"


# -----------------------------
# DIV2K split
# -----------------------------

div2k_train_clean_dir = div2k_clean_base_dir / "DIV2K_train_HR" / "DIV2K_train_HR"
div2k_valid_clean_dir = div2k_clean_base_dir / "DIV2K_valid_HR" / "DIV2K_valid_HR"

div2k_train_noisy_dir = div2k_noisy_base_dir / "DIV2K_train_HR" / "DIV2K_train_HR"
div2k_valid_noisy_dir = div2k_noisy_base_dir / "DIV2K_valid_HR" / "DIV2K_valid_HR"

In [5]:
def get_image_paths(input_dir):
    """
    Restituisce tutti i path delle immagini contenute in input_dir,
    includendo eventuali sottocartelle.
    """

    input_dir = Path(input_dir)

    image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]

    image_paths = [
        p for p in input_dir.rglob("*")
        if p.suffix.lower() in image_extensions
    ]

    return sorted(image_paths)

## Definizione dei metodi di denoising classici

In questa sezione vengono definite le funzioni per applicare i metodi classici di denoising.

I metodi considerati sono:

- filtro media;
- filtro mediano;
- BM3D.

Le immagini vengono gestite come array `float32` normalizzati nell'intervallo `[0, 1]`.  
Quando necessario, vengono temporaneamente convertite in `uint8` per l'utilizzo con OpenCV.

### Filtro media

Il filtro media sostituisce ogni pixel con la media dei valori presenti in una finestra locale.  
È un filtro semplice e può ridurre il rumore gaussiano, ma tende anche a sfocare bordi e dettagli fini dell'immagine.

In [6]:
def apply_mean_filter(noisy_np, kernel_size=3):
    """
    Applica un filtro media a un'immagine RGB float in [0,1].
    """
    # Conversione da float [0,1] a uint8 [0,255]
    noisy_uint8 = (np.clip(noisy_np, 0, 1) * 255).astype(np.uint8)
    # Applicazione del filtro media tramite OpenCV
    filtered_uint8 = cv2.blur(
        noisy_uint8,
        (kernel_size, kernel_size)
    )

    filtered_np = filtered_uint8.astype(np.float32) / 255.0

    return filtered_np

### Filtro mediano

Il filtro mediano sostituisce ogni pixel con la mediana dei valori presenti in una finestra locale.  
È particolarmente efficace contro rumori impulsivi, come salt-and-pepper, ma viene valutato anche sul rumore gaussiano come baseline classica semplice.

In [7]:
def apply_median_filter(noisy_np, kernel_size=3):
    """
    Applica un filtro mediano a un'immagine RGB float in [0,1].
    """

    noisy_uint8 = (np.clip(noisy_np, 0, 1) * 255).astype(np.uint8)

    filtered_uint8 = cv2.medianBlur(
        noisy_uint8,
        kernel_size
    )

    filtered_np = filtered_uint8.astype(np.float32) / 255.0

    return filtered_np

### BM3D

BM3D è un algoritmo classico di denoising basato sul raggruppamento di patch simili e sulla filtrazione collaborativa.  
È considerato una baseline molto forte per il rumore gaussiano ed è spesso utilizzato nella letteratura sul denoising.

Nel nostro caso, poiché le immagini sono RGB, BM3D viene applicato separatamente sui tre canali colore.

In [8]:
def apply_bm3d_rgb(noisy_np, sigma=25/255):
    """
    Applica BM3D a un'immagine RGB float in [0,1].
    Il filtro viene applicato separatamente sui tre canali.
    """
    # Array di output con la stessa shape dell'immagine di input
    denoised = np.zeros_like(noisy_np, dtype=np.float32)
    # Applicazione di BM3D canale per canale
    for c in range(3):
        denoised[:, :, c] = bm3d.bm3d(
            noisy_np[:, :, c],
            sigma_psd=sigma
        )

    denoised = np.clip(denoised, 0, 1)

    return denoised

## Funzione generale di valutazione

La funzione `evaluate_baseline_dataset` applica un metodo di denoising a tutte le immagini rumorose di un dataset, salva le immagini denoised e calcola le metriche quantitative rispetto alle immagini pulite.

Per ogni immagine vengono calcolati:

- PSNR clean/noisy;
- SSIM clean/noisy;
- PSNR clean/denoised;
- SSIM clean/denoised;
- gain PSNR;
- gain SSIM.

I risultati vengono salvati in un file CSV, così da poter essere analizzati e confrontati successivamente.

In [9]:
def evaluate_baseline_dataset(
    method_name,
    filter_function,
    noisy_dir,
    clean_dir,
    output_dir,
    csv_name="metrics.csv"
):
    """
    Applica un metodo di denoising classico a tutte le immagini rumorose
    contenute in noisy_dir, salva gli output e calcola PSNR/SSIM rispetto
    alle immagini clean corrispondenti.
    """

    noisy_dir = Path(noisy_dir)
    clean_dir = Path(clean_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    noisy_paths = get_image_paths(noisy_dir)

    results = []

    for noisy_path in tqdm(noisy_paths, desc=method_name):
        try:
            relative_path = noisy_path.relative_to(noisy_dir)
            clean_path = clean_dir / relative_path

            if not clean_path.exists():
                print("Clean non trovata:", clean_path)
                continue

            # Lettura noisy
            noisy_img = Image.open(noisy_path).convert("RGB")
            noisy_np = np.array(noisy_img).astype(np.float32) / 255.0

            # Lettura clean
            clean_img = Image.open(clean_path).convert("RGB")
            clean_img = clean_img.resize((noisy_np.shape[1], noisy_np.shape[0]))
            clean_np = np.array(clean_img).astype(np.float32) / 255.0

            # Applicazione filtro
            denoised_np = filter_function(noisy_np)
            denoised_np = np.clip(denoised_np, 0, 1)

            # Salvataggio immagine denoised
            save_path = output_dir / relative_path
            save_path = save_path.with_suffix(".png")
            save_path.parent.mkdir(parents=True, exist_ok=True)

            Image.fromarray(
                (denoised_np * 255).astype(np.uint8)
            ).save(save_path)

            # Metriche noisy
            psnr_noisy = psnr(clean_np, noisy_np, data_range=1.0)
            ssim_noisy = ssim(clean_np, noisy_np, channel_axis=2, data_range=1.0)

            # Metriche metodo
            psnr_denoised = psnr(clean_np, denoised_np, data_range=1.0)
            ssim_denoised = ssim(clean_np, denoised_np, channel_axis=2, data_range=1.0)

            results.append({
                "method": method_name,
                "filename": str(relative_path),
                "clean_path": str(clean_path),
                "noisy_path": str(noisy_path),
                "denoised_path": str(save_path),
                "psnr_clean_noisy": float(psnr_noisy),
                "ssim_clean_noisy": float(ssim_noisy),
                "psnr_clean_denoised": float(psnr_denoised),
                "ssim_clean_denoised": float(ssim_denoised),
                "psnr_gain": float(psnr_denoised - psnr_noisy),
                "ssim_gain": float(ssim_denoised - ssim_noisy),
            })

        except Exception as e:
            print("Errore su:", noisy_path)
            print(e)

    df = pd.DataFrame(results)

    csv_path = output_dir / csv_name
    df.to_csv(csv_path, index=False)

    print("\nMetodo:", method_name)
    print("CSV salvato in:", csv_path)

    if len(df) > 0:
        print("PSNR medio clean/noisy:", df["psnr_clean_noisy"].mean())
        print("PSNR medio clean/denoised:", df["psnr_clean_denoised"].mean())
        print("Gain PSNR medio:", df["psnr_gain"].mean())

        print("SSIM medio clean/noisy:", df["ssim_clean_noisy"].mean())
        print("SSIM medio clean/denoised:", df["ssim_clean_denoised"].mean())
        print("Gain SSIM medio:", df["ssim_gain"].mean())

    return df

## Esperimenti su Kodak
In questa sezione vengono applicati:

- filtro media 3x3;
- filtro mediano 3x3;
- BM3D.

In [10]:
df_kodak_mean3 = evaluate_baseline_dataset(
    method_name="mean_3x3",
    filter_function=lambda img: apply_mean_filter(img, kernel_size=3),
    noisy_dir=kodak_noisy_dir,
    clean_dir=kodak_clean_dir,
    output_dir=output_base_dir / "kodak" / "mean_3x3",
    csv_name="kodak_gaussian_sigma25_mean_3x3_metrics.csv"
)

mean_3x3: 100%|██████████| 24/24 [00:07<00:00,  3.02it/s]


Metodo: mean_3x3
CSV salvato in: /kaggle/working/gaussian_sigma25_classical_baselines/kodak/mean_3x3/kodak_gaussian_sigma25_mean_3x3_metrics.csv
PSNR medio clean/noisy: 20.438355583664833
PSNR medio clean/denoised: 25.946637321844538
Gain PSNR medio: 5.508281738179704
SSIM medio clean/noisy: 0.3498994478334983
SSIM medio clean/denoised: 0.624384139974912
Gain SSIM medio: 0.27448469400405884


## Risultati Kodak->Filtro Media

In [11]:
df_kodak_median3 = evaluate_baseline_dataset(
    method_name="median_3x3",
    filter_function=lambda img: apply_median_filter(img, kernel_size=3),
    noisy_dir=kodak_noisy_dir,
    clean_dir=kodak_clean_dir,
    output_dir=output_base_dir / "kodak" / "median_3x3",
    csv_name="kodak_gaussian_sigma25_median_3x3_metrics.csv"
)

median_3x3: 100%|██████████| 24/24 [00:08<00:00,  2.89it/s]


Metodo: median_3x3
CSV salvato in: /kaggle/working/gaussian_sigma25_classical_baselines/kodak/median_3x3/kodak_gaussian_sigma25_median_3x3_metrics.csv
PSNR medio clean/noisy: 20.438355583664833
PSNR medio clean/denoised: 25.437340142165084
Gain PSNR medio: 4.998984558500255
SSIM medio clean/noisy: 0.3498994478334983
SSIM medio clean/denoised: 0.559254739433527
Gain SSIM medio: 0.20935529160002866


## Risultati Kodak->filtro mediano

In [12]:
df_kodak_bm3d = evaluate_baseline_dataset(
    method_name="bm3d",
    filter_function=lambda img: apply_bm3d_rgb(img, sigma=25/255),
    noisy_dir=kodak_noisy_dir,
    clean_dir=kodak_clean_dir,
    output_dir=output_base_dir / "kodak" / "bm3d",
    csv_name="kodak_gaussian_sigma25_bm3d_metrics.csv"
)

bm3d: 100%|██████████| 24/24 [10:52<00:00, 27.20s/it]


Metodo: bm3d
CSV salvato in: /kaggle/working/gaussian_sigma25_classical_baselines/kodak/bm3d/kodak_gaussian_sigma25_bm3d_metrics.csv
PSNR medio clean/noisy: 20.438355583664833
PSNR medio clean/denoised: 29.6103641865249
Gain PSNR medio: 9.172008602860071
SSIM medio clean/noisy: 0.3498994478334983
SSIM medio clean/denoised: 0.8066062554717064
Gain SSIM medio: 0.45670681322614354


## Risultati Kodak->BM3D

## Esperimenti su BSD500
In questa sezione vengono applicati:

- filtro media 3x3;
- filtro mediano 3x3;
- BM3D.

In [13]:
df_bsd500_mean3 = evaluate_baseline_dataset(
    method_name="mean_3x3",
    filter_function=lambda img: apply_mean_filter(img, kernel_size=3),
    noisy_dir=bsd500_test_noisy_dir,
    clean_dir=bsd500_test_clean_dir,
    output_dir=output_base_dir / "bsd500_test" / "mean_3x3",
    csv_name="bsd500_test_gaussian_sigma25_mean_3x3_metrics.csv"
)

mean_3x3: 100%|██████████| 200/200 [00:21<00:00,  9.24it/s]


Metodo: mean_3x3
CSV salvato in: /kaggle/working/gaussian_sigma25_classical_baselines/bsd500_test/mean_3x3/bsd500_test_gaussian_sigma25_mean_3x3_metrics.csv
PSNR medio clean/noisy: 23.411698359831433
PSNR medio clean/denoised: 25.975743387635674
Gain PSNR medio: 2.5640450278042395
SSIM medio clean/noisy: 0.5265864753723144
SSIM medio clean/denoised: 0.6953984072804451
Gain SSIM medio: 0.16881193190813065


## Risultati BSD500->Filtro Media

In [14]:
df_bsd500_median3 = evaluate_baseline_dataset(
    method_name="median_3x3",
    filter_function=lambda img: apply_median_filter(img, kernel_size=3),
    noisy_dir=bsd500_test_noisy_dir,
    clean_dir=bsd500_test_clean_dir,
    output_dir=output_base_dir / "bsd500_test" / "median_3x3",
    csv_name="bsd500_test_gaussian_sigma25_median_3x3_metrics.csv"
)

median_3x3: 100%|██████████| 200/200 [00:21<00:00,  9.19it/s]


Metodo: median_3x3
CSV salvato in: /kaggle/working/gaussian_sigma25_classical_baselines/bsd500_test/median_3x3/bsd500_test_gaussian_sigma25_median_3x3_metrics.csv
PSNR medio clean/noisy: 23.411698359831433
PSNR medio clean/denoised: 25.930216800889863
Gain PSNR medio: 2.5185184410584243
SSIM medio clean/noisy: 0.5265864753723144
SSIM medio clean/denoised: 0.6648307311534881
Gain SSIM medio: 0.13824425578117372


## Risultati BSD500->Filtro Mediano

In [15]:
df_bsd500_bm3d = evaluate_baseline_dataset(
    method_name="bm3d",
    filter_function=lambda img: apply_bm3d_rgb(img, sigma=25/255),
    noisy_dir=bsd500_test_noisy_dir,
    clean_dir=bsd500_test_clean_dir,
    output_dir=output_base_dir / "bsd500_test" / "bm3d",
    csv_name="bsd500_test_gaussian_sigma25_bm3d_metrics.csv"
)

bm3d: 100%|██████████| 200/200 [36:36<00:00, 10.98s/it]


Metodo: bm3d
CSV salvato in: /kaggle/working/gaussian_sigma25_classical_baselines/bsd500_test/bm3d/bsd500_test_gaussian_sigma25_bm3d_metrics.csv
PSNR medio clean/noisy: 23.411698359831433
PSNR medio clean/denoised: 28.50790240561013
Gain PSNR medio: 5.096204045778694
SSIM medio clean/noisy: 0.5265864753723144
SSIM medio clean/denoised: 0.798812752366066
Gain SSIM medio: 0.2722262766957283


## Risultati BSD500->BM3D

## Esperimenti su DIV2K
In questa sezione vengono applicati:

- filtro media 3x3;
- filtro mediano 3x3;
- BM3D.

In [16]:
df_div2k_mean5 = evaluate_baseline_dataset(
    method_name="mean_5x5",
    filter_function=lambda img: apply_mean_filter(img, kernel_size=5),
    noisy_dir=div2k_valid_noisy_dir,
    clean_dir=div2k_valid_clean_dir,
    output_dir=output_base_dir / "div2k_valid" / "mean_5x5",
    csv_name="div2k_valid_gaussian_sigma25_mean_5x5_metrics.csv"
)

mean_5x5: 100%|██████████| 100/100 [04:25<00:00,  2.66s/it]


Metodo: mean_5x5
CSV salvato in: /kaggle/working/gaussian_sigma25_classical_baselines/div2k_valid/mean_5x5/div2k_valid_gaussian_sigma25_mean_5x5_metrics.csv
PSNR medio clean/noisy: 20.70517965059626
PSNR medio clean/denoised: 25.92011138117592
Gain PSNR medio: 5.214931730579666
SSIM medio clean/noisy: 0.35010348752140996
SSIM medio clean/denoised: 0.673810964524746
Gain SSIM medio: 0.32370747670531275


## Risultati DIV2K->Filtro Media

In [17]:
df_div2k_median3 = evaluate_baseline_dataset(
    method_name="median_3x3",
    filter_function=lambda img: apply_median_filter(img, kernel_size=3),
    noisy_dir=div2k_valid_noisy_dir,
    clean_dir=div2k_valid_clean_dir,
    output_dir=output_base_dir / "div2k_valid" / "median_3x3",
    csv_name="div2k_valid_gaussian_sigma25_median_3x3_metrics.csv"
)

median_3x3: 100%|██████████| 100/100 [03:30<00:00,  2.10s/it]


Metodo: median_3x3
CSV salvato in: /kaggle/working/gaussian_sigma25_classical_baselines/div2k_valid/median_3x3/div2k_valid_gaussian_sigma25_median_3x3_metrics.csv
PSNR medio clean/noisy: 20.70517965059626
PSNR medio clean/denoised: 25.895093455503133
Gain PSNR medio: 5.1899138049068805
SSIM medio clean/noisy: 0.35010348752140996
SSIM medio clean/denoised: 0.5792571917176247
Gain SSIM medio: 0.22915370479226113


## Risultati DIV2K->Filtro Mediano

In [18]:
df_div2k_bm3d = evaluate_baseline_dataset(
    method_name="bm3d",
    filter_function=lambda img: apply_bm3d_rgb(img, sigma=25/255),
    noisy_dir=div2k_valid_noisy_dir,
    clean_dir=div2k_valid_clean_dir,
    output_dir=output_base_dir / "div2k_valid" / "bm3d",
    csv_name="div2k_valid_gaussian_sigma25_bm3d_metrics.csv"
)

bm3d: 100%|██████████| 100/100 [5:11:34<00:00, 186.95s/it]


Metodo: bm3d
CSV salvato in: /kaggle/working/gaussian_sigma25_classical_baselines/div2k_valid/bm3d/div2k_valid_gaussian_sigma25_bm3d_metrics.csv
PSNR medio clean/noisy: 20.70517965059626
PSNR medio clean/denoised: 30.20533746065775
Gain PSNR medio: 9.500157810061488
SSIM medio clean/noisy: 0.35010348752140996
SSIM medio clean/denoised: 0.8290764489769935
Gain SSIM medio: 0.47897296264767647


## Risultati DIV2K->BM3D